In [1]:
import pandas as pd
import time
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

from heart import config as cfg
from heart.data import load_raw, make_splits
from heart.preprocessing import build_preprocessor
from heart.models import build_model, get_specs

In [2]:
# Load data
df = load_raw()
X_train, X_test, y_train, y_test = make_splits(df)

Calculating Metrics: 100%|██████████| 123/123 [00:00<00:00, 2730.23it/s]

Data validation PASSED: 36/36 checks successful


In [3]:
# Check the splits work
print("rows:", len(X_train), len(X_test))
print("disease rate:", y_train.mean().round(3), y_test.mean().round(3))
print("zero-chol share:", (X_train["Cholesterol"] == 0).mean().round(3),
      (X_test["Cholesterol"] == 0).mean().round(3))

# See if the features split evenly
Xt = build_preprocessor().fit_transform(X_train)
print(Xt.shape, int(Xt["chol_missing"].sum()))
Xt.describe()

rows: 734 184
disease rate: 0.553 0.554
zero-chol share: 0.187 0.19
(734, 19) 137


,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,Sex_M,ExerciseAngina_Y,FastingBS_1,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,chol_missing
count,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000
mean,-0.065348,0.093099,0.049555,-0.021316,0.011750,0.780654,0.405995,0.232970,0.539510,0.188011,0.217984,0.054496,0.196185,0.613079,0.190736,0.070845,0.501362,0.427793,0.186649
std,0.724924,0.832185,0.885860,0.716954,0.622283,0.414086,0.491418,0.423012,0.498776,0.390987,0.413158,0.227148,0.397381,0.487377,0.393149,0.256740,0.500339,0.495096,0.389895
min,-1.848126,-2.283505,-2.794141,-1.856291,-4.092509,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.538272,-0.514587,-0.445753,-0.505720,-0.544627,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,0.461728,0.485413,0.554247,0.494280,0.455373,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,1.852728,1.799157,2.340342,1.713941,1.343625,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
# Metrics and cross validation
SCORING = {
    "accuracy": "accuracy",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=cfg.SEED)

fold_scores = {}  

In [5]:
# Function to run the cross validation and return scores
def run_cv(name, model):
    # Cross-validate on the training set and store the per-fold scores.
    raw = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING,
                         return_train_score=True, n_jobs=-1)
    scores = pd.DataFrame({k: v for k, v in raw.items()
                           if k.startswith(("test_", "train_"))})
    scores[["test_brier", "train_brier"]] *= -1  # back to "lower is better"
    fold_scores[name] = scores

In [6]:
# Function to summarize metrics per model
def summary():
    # One row per model: validation mean ± std, plus the train/validation ROC AUC gap.
    metrics = ["recall", "precision", "f1", "roc_auc", "pr_auc", "brier", "accuracy"]
    rows = {}
    for name, s in fold_scores.items():
        row = {m: f"{s['test_' + m].mean():.3f} ± {s['test_' + m].std():.3f}"
               for m in metrics}
        row["auc_gap"] = round(s["train_roc_auc"].mean() - s["test_roc_auc"].mean(), 3)
        rows[name] = row
    return pd.DataFrame(rows).T

In [7]:
# Run for the 3 baseline models, dummy majority, dummy startified and logistic regression
baselines = {
    "dummy_majority": DummyClassifier(strategy="most_frequent"),
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=cfg.SEED),
    "logreg": make_pipeline(build_preprocessor(), LogisticRegression(max_iter=1000)),
}

for name, model in baselines.items():
    run_cv(name, model)

summary()

,recall,precision,f1,roc_auc,pr_auc,brier,accuracy,auc_gap
dummy_majority,1.000 ± 0.000,0.553 ± 0.003,0.712 ± 0.002,0.500 ± 0.000,0.553 ± 0.003,0.447 ± 0.003,0.553 ± 0.003,0.0
dummy_stratified,0.604 ± 0.034,0.557 ± 0.033,0.580 ± 0.033,0.505 ± 0.038,0.557 ± 0.020,0.484 ± 0.038,0.516 ± 0.038,-0.009
logreg,0.899 ± 0.030,0.865 ± 0.027,0.881 ± 0.020,0.926 ± 0.017,0.932 ± 0.018,0.103 ± 0.014,0.866 ± 0.023,0.013


In [ ]:
# Start over and rank all models to find the best ones based on roc_auc
fold_scores = {}  
for name in get_specs():
    start = time.time()
    run_cv(name, build_model(name))
    print(f"{name:18s} {time.time() - start:5.1f}s")

# Sort by mean validation ROC AUC
order = sorted(fold_scores, key=lambda n: -fold_scores[n]["test_roc_auc"].mean())
summary().loc[order]

dummy_majority       2.2s
dummy_stratified     2.1s
logreg               2.6s
knn                  2.6s
decision_tree        1.5s
random_forest        9.3s
xgboost              2.6s
lightgbm             3.4s
catboost             5.6s


,recall,precision,f1,roc_auc,pr_auc,brier,accuracy,auc_gap
logreg,0.899 ± 0.030,0.865 ± 0.027,0.881 ± 0.020,0.926 ± 0.017,0.932 ± 0.018,0.103 ± 0.014,0.866 ± 0.023,0.013
catboost,0.905 ± 0.029,0.871 ± 0.027,0.887 ± 0.018,0.924 ± 0.018,0.918 ± 0.025,0.100 ± 0.014,0.873 ± 0.021,0.072
random_forest,0.914 ± 0.028,0.859 ± 0.025,0.885 ± 0.016,0.924 ± 0.017,0.920 ± 0.024,0.103 ± 0.011,0.869 ± 0.018,0.076
knn,0.903 ± 0.029,0.861 ± 0.022,0.881 ± 0.017,0.922 ± 0.016,0.914 ± 0.021,0.103 ± 0.011,0.865 ± 0.019,0.023
xgboost,0.895 ± 0.029,0.869 ± 0.025,0.881 ± 0.019,0.922 ± 0.018,0.918 ± 0.024,0.102 ± 0.014,0.867 ± 0.022,0.072
lightgbm,0.891 ± 0.029,0.863 ± 0.026,0.876 ± 0.016,0.917 ± 0.017,0.914 ± 0.022,0.111 ± 0.016,0.861 ± 0.018,0.083
decision_tree,0.876 ± 0.046,0.835 ± 0.026,0.855 ± 0.028,0.882 ± 0.030,0.870 ± 0.034,0.128 ± 0.020,0.835 ± 0.030,0.056
dummy_stratified,0.604 ± 0.034,0.557 ± 0.033,0.580 ± 0.033,0.505 ± 0.038,0.557 ± 0.020,0.484 ± 0.038,0.516 ± 0.038,-0.009
dummy_majority,1.000 ± 0.000,0.553 ± 0.003,0.712 ± 0.002,0.500 ± 0.000,0.553 ± 0.003,0.447 ± 0.003,0.553 ± 0.003,0.0


Best Recall: random forest, catboost, knn
Best Roc: log reg, catboost/forest, knn/xgb
Best accuracy: catboost, forest, xgb
best auc_gap: logreg, knn, catboost